In [ ]:
from pyspark.sql import functions as F

# ============================================================
# RELOAD SAVED SILVER TABLES
# ============================================================

df_silver_final = spark.table(
    "Silver_HCC.dbo.silver_member_risk_final"
)

df_v22_mapping = spark.table(
    "Silver_HCC.dbo.silver_hcc_v22_mapping"
)

df_v28_mapping = spark.table(
    "Silver_HCC.dbo.silver_hcc_v28_mapping"
)

df_diagnosis_hcc = spark.table(
    "Silver_HCC.dbo.silver_diagnosis_hcc"
)


# ============================================================
# RECREATE UNIQUE MEMBER-HCC DATAFRAMES
# ============================================================

df_member_v22_unique = (
    df_diagnosis_hcc
    .filter(F.col("hcc_v22").isNotNull())
    .select(
        "member_id",
        F.col("hcc_v22").cast("int").alias("hcc_v22")
    )
    .dropDuplicates([
        "member_id",
        "hcc_v22"
    ])
)

df_member_v28_unique = (
    df_diagnosis_hcc
    .filter(F.col("hcc_v28").isNotNull())
    .select(
        "member_id",
        F.col("hcc_v28").cast("int").alias("hcc_v28")
    )
    .dropDuplicates([
        "member_id",
        "hcc_v28"
    ])
)


# ============================================================
# QUICK CHECK
# ============================================================

print(
    "Silver final members:",
    df_silver_final.count()
)

print(
    "V22 mapping rows:",
    df_v22_mapping.count()
)

print(
    "V28 mapping rows:",
    df_v28_mapping.count()
)

In [ ]:

# ============================================================
# DIM_MEMBER
# ============================================================

df_dim_member = (
    df_silver_final
    .select(
        F.col("member_id").alias("member_key"),
        "member_id",
        "reference_year",
        "age",
        "age_band",
        "gender",
        "chronic_condition_count"
    )
    .dropDuplicates(["member_id"])
)


# ============================================================
# DIM_MODEL
# ============================================================

df_dim_model = spark.createDataFrame(
    [
        (
            22,
            "CMS-HCC V22",
            "V22",
            "CMS-HCC",
            "Standardized comparative scoring"
        ),
        (
            28,
            "CMS-HCC V28",
            "V28",
            "CMS-HCC",
            "Standardized comparative scoring"
        )
    ],
    [
        "model_key",
        "model_name",
        "model_version",
        "model_family",
        "scoring_method"
    ]
)


# ============================================================
# DIM_HCC - V22
# ============================================================

df_hcc_v22 = (
    df_v22_mapping
    .filter(F.col("hcc_v22").isNotNull())
    .groupBy(
        F.col("hcc_v22")
        .cast("int")
        .alias("hcc_code")
    )
    .agg(
        F.first(
            "description",
            ignorenulls=True
        ).alias("hcc_description")
    )
    .withColumn("model_key", F.lit(22))
    .withColumn("model_version", F.lit("V22"))
)


# ============================================================
# DIM_HCC - V28
# ============================================================

df_hcc_v28 = (
    df_v28_mapping
    .filter(F.col("hcc_v28").isNotNull())
    .groupBy(
        F.col("hcc_v28")
        .cast("int")
        .alias("hcc_code")
    )
    .agg(
        F.first(
            "description",
            ignorenulls=True
        ).alias("hcc_description")
    )
    .withColumn("model_key", F.lit(28))
    .withColumn("model_version", F.lit("V28"))
)


# ============================================================
# FINAL DIM_HCC
# ============================================================

df_dim_hcc = (
    df_hcc_v22
    .unionByName(df_hcc_v28)
    .withColumn(
        "hcc_key",
        F.concat(
            F.col("model_version"),
            F.lit("_HCC"),
            F.col("hcc_code").cast("string")
        )
    )
    .select(
        "hcc_key",
        "model_key",
        "model_version",
        "hcc_code",
        "hcc_description"
    )
)

print("dim_member:", df_dim_member.count())
print("dim_model :", df_dim_model.count())
print("dim_hcc   :", df_dim_hcc.count())


In [ ]:
# ============================================================
# FACT_MEMBER_RISK - V22
# One row per MEMBER + MODEL
# ============================================================

df_fact_risk_v22 = (
    df_silver_final
    .select(
        F.col("member_id").alias("member_key"),
        F.lit(22).alias("model_key"),

        F.col("risk_score_v22_std")
        .alias("risk_score_std"),

        F.col("hcc_count_v22")
        .alias("mapped_hcc_count"),

        F.col("payment_hcc_count_v22")
        .alias("payment_hcc_count"),

        F.when(
            F.col("risk_score_v22_std") < 1,
            "LOW"
        )
        .when(
            F.col("risk_score_v22_std") < 2,
            "MODERATE"
        )
        .when(
            F.col("risk_score_v22_std") < 4,
            "HIGH"
        )
        .otherwise("VERY_HIGH")
        .alias("risk_tier"),

        F.col("scoring_status")
    )
)


# ============================================================
# FACT_MEMBER_RISK - V28
# ============================================================

df_fact_risk_v28 = (
    df_silver_final
    .select(
        F.col("member_id").alias("member_key"),
        F.lit(28).alias("model_key"),

        F.col("risk_score_v28_std")
        .alias("risk_score_std"),

        F.col("hcc_count_v28")
        .alias("mapped_hcc_count"),

        F.col("payment_hcc_count_v28")
        .alias("payment_hcc_count"),

        F.when(
            F.col("risk_score_v28_std") < 1,
            "LOW"
        )
        .when(
            F.col("risk_score_v28_std") < 2,
            "MODERATE"
        )
        .when(
            F.col("risk_score_v28_std") < 4,
            "HIGH"
        )
        .otherwise("VERY_HIGH")
        .alias("risk_tier"),

        F.col("scoring_status")
    )
)


df_fact_member_risk = (
    df_fact_risk_v22
    .unionByName(df_fact_risk_v28)
)


# ============================================================
# FACT_MEMBER_HCC - V22
# Pre-hierarchy diagnosis → HCC assignments
# ============================================================

df_fact_hcc_v22 = (
    df_member_v22_unique
    .select(
        F.col("member_id").alias("member_key"),

        F.lit(22)
        .alias("model_key"),

        F.col("hcc_v22")
        .cast("int")
        .alias("hcc_code")
    )
    .withColumn(
        "hcc_key",
        F.concat(
            F.lit("V22_HCC"),
            F.col("hcc_code").cast("string")
        )
    )
    .withColumn(
        "assignment_stage",
        F.lit("DIAGNOSIS_MAPPING_PRE_HIERARCHY")
    )
)


# ============================================================
# FACT_MEMBER_HCC - V28
# ============================================================

df_fact_hcc_v28 = (
    df_member_v28_unique
    .select(
        F.col("member_id").alias("member_key"),

        F.lit(28)
        .alias("model_key"),

        F.col("hcc_v28")
        .cast("int")
        .alias("hcc_code")
    )
    .withColumn(
        "hcc_key",
        F.concat(
            F.lit("V28_HCC"),
            F.col("hcc_code").cast("string")
        )
    )
    .withColumn(
        "assignment_stage",
        F.lit("DIAGNOSIS_MAPPING_PRE_HIERARCHY")
    )
)


df_fact_member_hcc = (
    df_fact_hcc_v22
    .unionByName(df_fact_hcc_v28)
    .dropDuplicates([
        "member_key",
        "model_key",
        "hcc_code"
    ])
)


print(
    "fact_member_risk:",
    df_fact_member_risk.count()
)

print(
    "fact_member_hcc:",
    df_fact_member_hcc.count()
)

In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants


# ============================================================
# WRITE DIMENSIONS
# ============================================================

(
    df_dim_member
    .write
    .mode("overwrite")
    .synapsesql(
        "Gold_HCC.dbo.dim_member"
    )
)

(
    df_dim_model
    .write
    .mode("overwrite")
    .synapsesql(
        "Gold_HCC.dbo.dim_model"
    )
)

(
    df_dim_hcc
    .write
    .mode("overwrite")
    .synapsesql(
        "Gold_HCC.dbo.dim_hcc"
    )
)


# ============================================================
# WRITE FACTS
# ============================================================

(
    df_fact_member_risk
    .write
    .mode("overwrite")
    .synapsesql(
        "Gold_HCC.dbo.fact_member_risk"
    )
)

(
    df_fact_member_hcc
    .write
    .mode("overwrite")
    .synapsesql(
        "Gold_HCC.dbo.fact_member_hcc"
    )
)


# ============================================================
# VALIDATION
# ============================================================

tables = [
    "dim_member",
    "dim_model",
    "dim_hcc",
    "fact_member_risk",
    "fact_member_hcc"
]

for table in tables:

    count = (
        spark.read
        .synapsesql(
            f"Gold_HCC.dbo.{table}"
        )
        .count()
    )

    print(
        f"{table}: {count:,}"
    )

In [ ]:
print(
    "Duplicate member keys:",
    df_dim_member.groupBy("member_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate member/model risk rows:",
    df_fact_member_risk
    .groupBy("member_key", "model_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate member/model/HCC rows:",
    df_fact_member_hcc
    .groupBy("member_key", "model_key", "hcc_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)